# Grounding DINO auto-labelling
Colab GPU workflow exported from the original script. Review every cell before running.
Mount your own Drive, replace the author-specific ZIP/image paths and review the output labels.
This notebook restores the original directory magics; it has not been executed against a GPU in this change.
Model/code downloads require network access; Colab and Drive are cloud services, not a local-only workflow.


In [ ]:
%cd /content
# -*- coding: utf-8 -*-
"""GROUNDING DINO AUTO LABEL.ipynb

Automatically generated by Colaboratory.

Original file is located at
    https://colab.research.google.com/drive/1HE7DKEwAgrT0OLAydvKu-kS6ug_-KvqG
"""

# Commented out IPython magic to ensure Python compatibility.
!git clone https://github.com/IDEA-Research/GroundingDINO.git
%cd GroundingDINO
!git checkout -q 57535c5a79791cb76e36fdb64975271354f10251
!pip install -q -e .

from google.colab import drive
drive.mount('/content/drive')

!pip uninstall -y supervision
!pip install -q supervision==0.4.0

import supervision as svn
print(svn.__version__)

# Commented out IPython magic to ensure Python compatibility.
import os

GROUNDING_DINO_CONFIG_PATH = os.path.join("groundingdino/config/GroundingDINO_SwinT_OGC.py")
print(GROUNDING_DINO_CONFIG_PATH, "; exist:", os.path.isfile(GROUNDING_DINO_CONFIG_PATH))
!mkdir -p weights
%cd weights

!wget -q https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
import os
%cd /content/GroundingDINO
GROUNDING_DINO_CHECKPOINT_PATH = os.path.join("weights/groundingdino_swint_ogc.pth")
print(GROUNDING_DINO_CHECKPOINT_PATH, "; exist:", os.path.isfile(GROUNDING_DINO_CHECKPOINT_PATH))

import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from groundingdino.util.inference import Model

grounding_dino_model = Model(model_config_path=GROUNDING_DINO_CONFIG_PATH, model_checkpoint_path=GROUNDING_DINO_CHECKPOINT_PATH)

!mkdir -p data

!unzip /content/drive/MyDrive/test.zip -d /content/GroundingDINO/data

SOURCE_IMAGE_PATH = "/content/GroundingDINO/data/test/synergy_27082023_rr_10.jpg"
CLASSES = ['person','helmet','vibrocharger'] #add the class name to be labeled automatically
BOX_TRESHOLD = 0.34
TEXT_TRESHOLD = 0.15

# Commented out IPython magic to ensure Python compatibility.
from typing import List

def enhance_class_name(class_names: List[str]) -> List[str]:
   return [
       f"all {class_name}s"
       for class_name
       in class_names
   ]
import cv2
import supervision as sv

# load image
image = cv2.imread(SOURCE_IMAGE_PATH)

# detect objects
detections = grounding_dino_model.predict_with_classes(
   image=image,
   classes=enhance_class_name(class_names=CLASSES),
   box_threshold=BOX_TRESHOLD,
   text_threshold=TEXT_TRESHOLD
)

# annotate image with detections
box_annotator = svn.BoxAnnotator()
print(detections,"____________________________")
labels = [f"{CLASSES[class_id]} {confidence:0.2f}"
   for _, confidence, class_id, _
in detections]
annotated_frame = box_annotator.annotate(scene=image.copy(), detections=detections, labels=labels)

%matplotlib inline
svn.plot_image(annotated_frame, (16, 16))

import os

IMAGES_DIRECTORY = "./data/test"
IMAGES_EXTENSIONS = ['jpg', 'jpeg', 'png']

CLASSES = ['person','helmet']
BOX_TRESHOLD = 0.35
TEXT_TRESHOLD = 0.15

import cv2
from tqdm.notebook import tqdm

images = {}
annotations = {}

image_paths = svn.list_files_with_extensions(
   directory=IMAGES_DIRECTORY,
   extensions=IMAGES_EXTENSIONS)

for image_path in tqdm(image_paths):
   image_name = image_path.name
   image_path = str(image_path)
   image = cv2.imread(image_path)

   detections = grounding_dino_model.predict_with_classes(
       image=image,
       classes=enhance_class_name(class_names=CLASSES),
       box_threshold=BOX_TRESHOLD,
       text_threshold=TEXT_TRESHOLD
   )
   detections = detections[detections.class_id != None]
   images[image_name] = image
   annotations[image_name] = detections

plot_images = []
plot_titles = []

box_annotator = svn.BoxAnnotator()
mask_annotator = svn.MaskAnnotator()

for image_name, detections in annotations.items():
   image = images[image_name]
   plot_images.append(image)
   plot_titles.append(image_name)

   labels = [
       f"{CLASSES[class_id]} {confidence:0.2f}"
       for _, _, confidence, class_id, _
       in detections]
   annotated_image = mask_annotator.annotate(scene=image.copy(), detections=detections)
   annotated_image = box_annotator.annotate(scene=annotated_image, detections=detections, labels=labels)
   plot_images.append(annotated_image)
   title = " ".join(set([
       CLASSES[class_id]
       for class_id
       in detections.class_id
   ]))
   plot_titles.append(title)

svn.plot_images_grid(images=plot_images, titles=plot_titles, grid_size=(len(annotations), 2), size=(2 * 4, len(annotations) * 4))

import locale
locale.getpreferredencoding = lambda: "UTF-8"

# Commented out IPython magic to ensure Python compatibility.
%cd /content/GroundingDINO
!mkdir annotations

ANNOTATIONS_DIRECTORY = "/content/GroundingDINO/annotations"

MIN_IMAGE_AREA_PERCENTAGE = 0.002
MAX_IMAGE_AREA_PERCENTAGE = 0.80
APPROXIMATION_PERCENTAGE = 0.75
svn.Dataset(
   classes=CLASSES,
   images=images,
   annotations=annotations
).as_pascal_voc(
   annotations_directory_path=ANNOTATIONS_DIRECTORY,
   min_image_area_percentage=MIN_IMAGE_AREA_PERCENTAGE,
   max_image_area_percentage=MAX_IMAGE_AREA_PERCENTAGE,
   approximation_percentage=APPROXIMATION_PERCENTAGE
)

!zip -r /content/GroundingDINO/annotations.zip /content/GroundingDINO/annotations
